# HVAC Prediction Pipeline Notebook\n
\n
This notebook combines the functionality of the various Python scripts in the project (`split_dataset.py`, `core_components.py`, `utils.py`, `tcn_model.py`, `hybrid_model.py`, `train.py`, `predict_hvac.py`, `optimize.py`, `ensemble_train.py`, `run_pipeline.py`) into a single workflow.\n
\n
**Sections:**\n
1.  **Imports:** Load all necessary libraries.\n
2.  **Configuration:** Set parameters for data paths, model hyperparameters, training settings, etc. (Replaces command-line arguments).\n
3.  **Utility Functions:** Define helper functions (`add_time_features`, `safe_literal_eval`).\n
4.  **Core Components:** Define Dataset, Models (LSTM, TCN, Hybrid), and sequence creation logic.\n
5.  **Data Splitting:** Function to split the dataset into training and testing sets.\n
6.  **Training:** Functions for loading data, building models, training, evaluation, and artifact logging.\n
7.  **Prediction:** Functions for loading trained models and making predictions on new data.\n
8.  **Optimization (Optuna):** Functions for hyperparameter optimization using Optuna.\n
9.  **Ensemble Training:** Logic for training multiple models with different seeds.\n
10. **Pipeline Execution:** Example workflows for running training, prediction, optimization, or ensembling.

## 1. Imports

In [2]:
import os
import re
import argparse  # Keep for structure, but replace usage
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.parametrizations import weight_norm
import joblib
from tqdm.notebook import tqdm # Use notebook-friendly tqdm
import random
import ast
import json
import optuna

## 2. Configuration
Define all parameters here. These replace the command-line arguments used in the original scripts.

In [3]:
# --- Configuration Object (Simulates argparse Namespace) ---
class Config:
    def __init__(self):
        # Data/Paths
        self.train_csv = 'dataset/train_split.csv'
        self.test_csv = 'dataset/test_split.csv' # Used in training evaluation
        self.predict_features_csv = 'dataset/test_features.csv' # Used for final prediction
        self.output_dir = 'models_lstm' # Base directory for saving models/scalers (LSTM specific)
        self.results_dir = 'results_lstm' # Base directory for saving results (LSTM specific)
        # Note: Filenames are often adapted dynamically within runs (training, ensemble, optuna)
        self.model_base_filename = 'lstm_hvac_best.pth'
        self.feature_scaler_base_filename = 'feature_scaler_lstm.save'
        self.target_scaler_base_filename = 'target_scaler_lstm.save'
        self.results_base_filename = 'test_set_eval_lstm.csv'
        self.submission_base_filename = 'submission_lstm.csv'

        # Model Hyperparameters (LSTM Only)
        self.seq_len = 16
        self.hidden_size = 128
        self.num_layers = 2
        self.dropout = 0.2

        # Training Hyperparameters
        self.batch_size = 64
        self.num_epochs = 50 # Max epochs
        self.lr = 1e-3
        self.loss_function = 'Huber' # 'MSE' or 'Huber'
        self.val_split_ratio = 0.15

        # Feature Engineering
        self.use_time_features = True

        # Training Enhancements
        self.use_lr_scheduler = True
        self.lr_scheduler_patience = 5
        self.lr_scheduler_factor = 0.5
        self.use_early_stopping = True
        self.early_stopping_patience = 10
        self.use_gradient_clipping = True
        self.gradient_clip_max_norm = 1.0

        # Prediction Specific
        self.training_run_id = None # SET THIS when running prediction based on a specific training run

        # Optimization Specific (Optuna)
        self.n_trials = 30
        self.study_name = "hvac_optimization_lstm_notebook"
        self.storage_url = None # e.g., "sqlite:///hvac_optuna_lstm_notebook.db"

        # Ensemble Specific
        self.random_seeds = [42, 123, 456, 789, 1011]
        self.ensemble_run_name = "hvac_ensemble_lstm_notebook"
        self.model_paths_filename = "ensemble_lstm_model_paths_notebook.json"

        # --- Dynamic names (set within functions like run_training_trial) ---
        # These are set based on trial/seed to avoid overwriting
        self.model_filename = self.model_base_filename
        self.feature_scaler_filename = self.feature_scaler_base_filename
        self.target_scaler_filename = self.target_scaler_base_filename
        self.results_filename = self.results_base_filename
        self.output_filename = self.submission_base_filename # For prediction output

    def __getitem__(self, key):
        # Allow dictionary-like access
        return getattr(self, key)

    def get(self, key, default=None):
        # Allow dictionary-like access with default
        return getattr(self, key, default)

    def __contains__(self, key):
        # Allow 'in' checks
        return hasattr(self, key)

# Instantiate the config
config = Config()

# Display config (optional)
import pprint
pprint.pprint(vars(config))

{'batch_size': 64,
 'dropout': 0.2,
 'early_stopping_patience': 10,
 'ensemble_run_name': 'hvac_ensemble_lstm_notebook',
 'feature_scaler_base_filename': 'feature_scaler_lstm.save',
 'feature_scaler_filename': 'feature_scaler_lstm.save',
 'gradient_clip_max_norm': 1.0,
 'hidden_size': 128,
 'loss_function': 'Huber',
 'lr': 0.001,
 'lr_scheduler_factor': 0.5,
 'lr_scheduler_patience': 5,
 'model_base_filename': 'lstm_hvac_best.pth',
 'model_filename': 'lstm_hvac_best.pth',
 'model_paths_filename': 'ensemble_lstm_model_paths_notebook.json',
 'n_trials': 30,
 'num_epochs': 50,
 'num_layers': 2,
 'output_dir': 'models_lstm',
 'output_filename': 'submission_lstm.csv',
 'predict_features_csv': 'dataset/test_features.csv',
 'random_seeds': [42, 123, 456, 789, 1011],
 'results_base_filename': 'test_set_eval_lstm.csv',
 'results_dir': 'results_lstm',
 'results_filename': 'test_set_eval_lstm.csv',
 'seq_len': 16,
 'storage_url': None,
 'study_name': 'hvac_optimization_lstm_notebook',
 'submissio

## 3. Utility Functions

In [4]:
# From utils.py

def add_time_features(df):
    """Adds time-based features to the DataFrame."""
    # Ensure 'date' column is datetime type
    if 'date' not in df.columns:
        raise ValueError("'date' column is required for time features but not found.")
    df['date'] = pd.to_datetime(df['date'])
    df['hour'] = df['date'].dt.hour
    df['dayofweek'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    # Optionally add sine/cosine transformations for cyclical features
    df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24.0)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12.0)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12.0)
    return df

# From utils.py (originally added in predict_hvac.py)
def safe_literal_eval(val):
    """Safely evaluate a string containing a Python literal (list, tuple, number, bool, string, etc.)."""
    try:
        # Handle boolean strings explicitly
        if isinstance(val, str):
            if val.lower() == 'true':
                return True
            if val.lower() == 'false':
                return False
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        # If it's not a literal, return the original string
        return val

## 4. Core Components

In [5]:
# From core_components.py

class HVACDataset(Dataset):
    def __init__(self, sequences, targets):
        self.X = sequences.astype(np.float32)
        # Ensure target is float32 and potentially reshape for loss function compatibility
        self.y = targets.astype(np.float32).reshape(-1, 1)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1, dropout=0.0):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Apply dropout between LSTM layers only if num_layers > 1
        lstm_dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=lstm_dropout
        )

        # Batch normalization after LSTM output (applied to hidden_size features)
        self.bn = nn.BatchNorm1d(hidden_size)

        # Dropout layer before final FC layer
        self.dropout = nn.Dropout(dropout)

        # Final fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        out, (hn, cn) = self.lstm(x)

        # Use the output of the last time step
        # out shape: (batch, seq_len, hidden_size)
        # We want the features from the last time step: (batch, hidden_size)
        last_time_step_out = out[:, -1, :]

        # Apply batch normalization
        # BatchNorm1d expects input of shape (N, C) or (N, C, L)
        # Here N=batch_size, C=hidden_size
        bn_out = self.bn(last_time_step_out)

        # Apply dropout
        dropout_out = self.dropout(bn_out)

        # Final prediction
        final_out = self.fc(dropout_out)

        return final_out

def create_sequences(data, seq_len=16):
    """
    Create input sequences and targets for time series forecasting.
    data: tuple (features, consumption),
          features: np.array shape (N, n_features)
          consumption: np.array shape (N,)
    Returns:
      X: np.array shape (num_samples, seq_len, n_features)
      y: np.array shape (num_samples, 1) - Target is consumption at step t + seq_len + 15
    Target is a single consumption value 16 time steps after the *end* of the input sequence.
    """
    features, consumption = data
    X, y = [], []
    N = len(consumption)
    # Adjust loop range to ensure target index i + seq_len + 15 is valid
    for i in range(N - seq_len - 15):
        X.append(features[i:i+seq_len])
        # Target is the single value at step i + seq_len + 15
        y.append(consumption[i+seq_len+15])
    # Return y as a column vector
    return np.array(X), np.array(y)

# TCNModel, TemporalBlock, HybridModel REMOVED

## 5. Data Splitting

In [6]:
# From split_dataset.py

def split_data(input_csv=os.path.join('dataset', 'train_dataset.csv'), 
                 train_out=os.path.join('dataset', 'train_split.csv'), 
                 test_out=os.path.join('dataset', 'test_split.csv'), 
                 split_ratio=0.8):
    """Splits the dataset based on time."""
    print(f"Splitting {input_csv}...")
    try:
        df = pd.read_csv(input_csv, parse_dates=['date'])
    except FileNotFoundError:
        print(f"Error: Input file not found at {input_csv}")
        return
        
    df = df.sort_values('date').reset_index(drop=True)
    
    # Compute split index
    split_idx = int(len(df) * split_ratio)
    train_df = df.iloc[:split_idx]
    test_df = df.iloc[split_idx:]
    
    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(train_out), exist_ok=True)
    os.makedirs(os.path.dirname(test_out), exist_ok=True)
    
    # Save files
    train_df.to_csv(train_out, index=False)
    test_df.to_csv(test_out, index=False)
    
    print(f'Saved training split ({len(train_df)} rows) to {train_out}')
    print(f'Saved test split ({len(test_df)} rows) to {test_out}')

# Example Usage (Uncomment to run)
# split_data()

## 6. Training

In [7]:
def load_series(csv_path, feature_cols, target_col, use_time_features=True):
    """Loads, sorts, and adds time features (if enabled) to the time series data."""
    df = pd.read_csv(csv_path, parse_dates=['date'])
    df = df.sort_values('date').reset_index(drop=True)

    if use_time_features:
        # Assumes add_time_features function is available
        df = add_time_features(df)
        time_feature_cols = ['hour_sin', 'hour_cos', 'dayofweek', 'month_sin', 'month_cos']
        all_feature_cols = feature_cols + time_feature_cols
    else:
        all_feature_cols = feature_cols

    features = df[all_feature_cols].values
    target = df[target_col].values
    return features, target, df, all_feature_cols

def load_and_prepare_data(args):
    """Loads train/test data, scales features/targets, creates sequences, and returns DataLoaders."""
    base_feature_cols = [
        'temperature_exterieure', 'humidite', 'ensoleillement',
        'vitesse_vent', 'direction_vent', 'temperature_interieure'
    ]
    target_col = 'puissance_cvac'

    # Load train data and get feature list
    feat_train_val, targ_train_val, _, all_feature_cols = load_series(
        args.train_csv, base_feature_cols, target_col, args.use_time_features
    )
    # Load test data using the same feature list
    feat_test, targ_test, _, _ = load_series(
        args.test_csv, base_feature_cols, target_col, args.use_time_features
    )

    # Scale Features
    feat_scaler = MinMaxScaler()
    feat_train_val_s = feat_scaler.fit_transform(feat_train_val)
    feat_test_s = feat_scaler.transform(feat_test)

    # Scale Targets
    targ_scaler = MinMaxScaler()
    targ_train_val_s = targ_scaler.fit_transform(targ_train_val.reshape(-1,1)).flatten()
    targ_test_s = targ_scaler.transform(targ_test.reshape(-1,1)).flatten()

    # Create Sequences (Assumes create_sequences function is available)
    X_train_val, y_train_val = create_sequences((feat_train_val_s, targ_train_val_s), seq_len=args.seq_len)
    X_test, y_test = create_sequences((feat_test_s, targ_test_s), seq_len=args.seq_len)

    # Split Train/Validation
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=args.val_split_ratio, shuffle=False
    )

    # DataLoaders (Assumes HVACDataset class is available)
    train_ds = HVACDataset(X_train, y_train)
    val_ds = HVACDataset(X_val, y_val)
    test_ds = HVACDataset(X_test, y_test)

    num_workers = 0 # Simplified
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, pin_memory=True, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, pin_memory=True, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=args.batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, feat_scaler, targ_scaler, all_feature_cols

def build_model(args, input_size, device):
    """Builds the LSTM model."""
    # Assumes LSTMModel class is available
    model = LSTMModel(
        input_size=input_size,
        hidden_size=args.hidden_size,
        num_layers=args.num_layers,
        dropout=args.dropout
    )
    model.to(device)
    return model

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, args, device):
    """Trains the model, performs validation, handles early stopping, and saves the best model."""
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_path = os.path.join(args.output_dir, args.model_filename)
    train_ds_len = len(train_loader.dataset)
    val_ds_len = len(val_loader.dataset)

    os.makedirs(args.output_dir, exist_ok=True) # Ensure directory exists

    pbar_epochs = tqdm(range(1, args.num_epochs + 1), desc='Epochs', leave=False)
    for epoch in pbar_epochs:
        # Training Phase
        model.train()
        train_loss = 0.0
        train_pbar = tqdm(train_loader, desc=f'Train Ep {epoch}', leave=False)
        for Xb, yb in train_pbar:
            Xb, yb = Xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            optimizer.zero_grad()
            out = model(Xb)
            loss = criterion(out, yb)
            loss.backward()
            if args.use_gradient_clipping:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=args.gradient_clip_max_norm)
            optimizer.step()
            train_loss += loss.item() * Xb.size(0)
            train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        avg_train_loss = train_loss / train_ds_len

        # Validation Phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
                out_scaled = model(Xb)
                loss = criterion(out_scaled, yb)
                val_loss += loss.item() * Xb.size(0)
        avg_val_loss = val_loss / val_ds_len

        pbar_epochs.set_description(f'Epoch {epoch}/{args.num_epochs} TrL:{avg_train_loss:.5f} Vl L:{avg_val_loss:.5f}')

        # LR Scheduling
        if args.use_lr_scheduler and scheduler:
            scheduler.step(avg_val_loss)

        # Early Stopping & Saving Best Model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), best_model_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if args.use_early_stopping and epochs_no_improve >= args.early_stopping_patience:
                break # Stop training early

    # Return path only if model file was actually created
    return best_model_path if os.path.exists(best_model_path) else None

def evaluate_model(model, best_model_path, test_loader, targ_scaler, device):
    """Evaluates the model on the test set using the saved best weights."""
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    model.eval()
    test_preds_scaled_list = []
    test_trues_scaled_list = []

    with torch.no_grad():
        for Xb, yb in tqdm(test_loader, desc='Evaluating', leave=False):
            Xb = Xb.to(device, non_blocking=True)
            out_scaled = model(Xb).cpu().numpy()
            test_preds_scaled_list.append(out_scaled)
            test_trues_scaled_list.append(yb.numpy())

    preds_scaled = np.concatenate(test_preds_scaled_list)
    trues_scaled = np.concatenate(test_trues_scaled_list)

    # Inverse transform to original scale
    preds = targ_scaler.inverse_transform(preds_scaled).flatten()
    trues = targ_scaler.inverse_transform(trues_scaled).flatten()

    # Calculate final metrics
    mse = mean_squared_error(trues, preds)
    mae = mean_absolute_error(trues, preds)

    results_df = pd.DataFrame({'y_true': trues, 'y_pred': preds})

    return mse, mae, results_df

def run_training_trial(args, device, trial_num=None, random_seed=None):
    """
    Saves model, scalers, and results to disk.
    Returns final test MAE and the path to the best model saved.
    """
    # --- Setup & Seeding ---
    current_seed = random_seed if random_seed is not None else random.randint(1, 10000)
    torch.manual_seed(current_seed)
    np.random.seed(current_seed)
    random.seed(current_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(current_seed)

    # --- File Naming ---
    output_dir = args.output_dir
    results_dir = args.results_dir
    # Use getattr with defaults for safety if args might be missing these
    model_base_filename = getattr(args, 'model_base_filename', 'model')
    feature_scaler_base_filename = getattr(args, 'feature_scaler_base_filename', 'feature_scaler')
    target_scaler_base_filename = getattr(args, 'target_scaler_base_filename', 'target_scaler')
    results_base_filename = getattr(args, 'results_base_filename', 'results')

    trial_suffix = f"_trial_{trial_num}" if trial_num is not None else ""
    seed_suffix = f"_seed_{random_seed if random_seed is not None else current_seed}"
    combined_suffix = f"{trial_suffix}{seed_suffix}" if trial_suffix or seed_suffix else "_run"

    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(results_dir, exist_ok=True)

    # Create a copy of args for this specific run to avoid modifying the original
    run_args = argparse.Namespace(**vars(args))
    run_args.model_filename = f"{os.path.splitext(model_base_filename)[0]}{combined_suffix}.pth"
    run_args.feature_scaler_filename = f"{os.path.splitext(feature_scaler_base_filename)[0]}{combined_suffix}.joblib" # Use joblib extension
    run_args.target_scaler_filename = f"{os.path.splitext(target_scaler_base_filename)[0]}{combined_suffix}.joblib" # Use joblib extension
    run_args.results_filename = f"{os.path.splitext(results_base_filename)[0]}{combined_suffix}.csv"
    if trial_num is not None:
        run_args.trial_num = trial_num # Keep track if needed downstream

    # --- Data Loading ---
    train_loader, val_loader, test_loader, feat_scaler, targ_scaler, all_feature_cols = load_and_prepare_data(run_args)

    # --- Model, Loss, Optimizer, Scheduler ---
    model = build_model(run_args, len(all_feature_cols), device)

    if run_args.loss_function == 'MSE':
        criterion = nn.MSELoss()
    elif run_args.loss_function == 'Huber':
        criterion = nn.HuberLoss(delta=getattr(run_args, 'huber_delta', 1.0)) # Allow delta configuration
    else:
        criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=run_args.lr)

    scheduler = None
    if run_args.use_lr_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min',
            factor=run_args.lr_scheduler_factor,
            patience=run_args.lr_scheduler_patience,
            verbose=False # No print statements
        )

    # --- Training ---
    best_model_path = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, run_args, device)

    # --- Evaluation & Saving Artifacts ---
    test_mse, test_mae, results_df = None, None, None
    optimization_metric = float('inf') # Default to infinity (bad score)

    if best_model_path:
        test_mse, test_mae, results_df = evaluate_model(model, best_model_path, test_loader, targ_scaler, device)

        if test_mae is not None: # Check if evaluation produced a valid metric
            # Save scalers
            feature_scaler_path = os.path.join(output_dir, run_args.feature_scaler_filename)
            target_scaler_path = os.path.join(output_dir, run_args.target_scaler_filename)
            joblib.dump(feat_scaler, feature_scaler_path)
            joblib.dump(targ_scaler, target_scaler_path)

            # Save evaluation results
            if results_df is not None:
                results_df_path = os.path.join(results_dir, run_args.results_filename)
                results_df.to_csv(results_df_path, index=False)

            optimization_metric = test_mae # Use MAE as the primary metric

    # Return the optimization metric (MAE or inf) and the model path (or None)
    final_model_path = best_model_path if optimization_metric != float('inf') else None
    return optimization_metric, final_model_path


## 7. Prediction

In [8]:
def prepare_test_data(predict_config, model_config):
    """
    Loads prediction features, adds time features if needed, scales features
    using a pre-trained scaler, and prepares sequences for the model.

    Args:
        predict_config: Config object containing paths like predict_features_csv,
                        feature_scaler_path.
        model_config: Config object containing model parameters like seq_len,
                      use_time_features, all_feature_cols, input_size.
    """
    test_csv_path = predict_config.predict_features_csv
    feature_scaler_local_path = predict_config.feature_scaler_path # Path to local scaler
    use_time_features = model_config.use_time_features
    all_feature_cols = model_config.all_feature_cols
    input_size = model_config.input_size
    seq_len = model_config.seq_len

    parse_dates = ['date'] if use_time_features else None
    df_test = pd.read_csv(test_csv_path, parse_dates=parse_dates)

    # Assume 'ID' column exists
    ids = df_test['ID'].values

    # --- Add Time Features (if applicable) ---
    if use_time_features:
        df_test = add_time_features(df_test) # Assumes utility function exists

    # --- Select Features ---
    # Assume required columns exist in the CSV
    features = df_test[all_feature_cols].values

    # --- Load Scaler and Scale Features ---
    # Assume scaler exists at the specified path and is compatible
    feat_scaler = joblib.load(feature_scaler_local_path)
    features_scaled = feat_scaler.transform(features)

    # --- Input Sequence Preparation ---
    # Simplified sequence creation for prediction (repeating last known features)
    num_samples = features_scaled.shape[0]
    num_features = features_scaled.shape[1]
    X_test_seq = np.zeros((num_samples, seq_len, num_features), dtype=np.float32)
    # Tile the single feature vector across the sequence length dimension
    for i in range(num_samples):
        X_test_seq[i, :, :] = np.tile(features_scaled[i, :], (seq_len, 1))

    return X_test_seq, ids

def build_model_from_params(model_config, device):
    """
    Builds the LSTM model structure based on provided configuration.

    Args:
        model_config: Config object containing model hyperparameters like
                      input_size, hidden_size, num_layers, dropout.
        device: The torch device (e.g., 'cuda' or 'cpu').
    """
    # Assumes LSTMModel class is available
    model = LSTMModel(
        input_size=model_config.input_size,
        hidden_size=model_config.hidden_size,
        num_layers=model_config.num_layers,
        dropout=model_config.dropout
    )
    model.to(device)
    return model

def run_prediction(model, X_test_seq, target_scaler_local_path, device, batch_size=256):
    """
    Runs predictions using the loaded model and data, and inverse transforms the results.

    Args:
        model: The loaded PyTorch model (weights should be loaded separately).
        X_test_seq: The prepared input data sequences.
        target_scaler_local_path: Path to the local target scaler file.
        device: The torch device.
        batch_size: Batch size for prediction.
    """
    model.eval() # Ensure model is in evaluation mode
    preds_scaled_list = []

    # Create DataLoader for batching predictions
    X_tensor = torch.from_numpy(X_test_seq).float()
    pred_dataset = TensorDataset(X_tensor)
    pred_loader = DataLoader(pred_dataset, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for Xb, in tqdm(pred_loader, desc="Predicting", leave=False):
            Xb = Xb.to(device)
            batch_preds_scaled = model(Xb).cpu().numpy() # Shape: (batch_size, 1)
            preds_scaled_list.append(batch_preds_scaled)

    preds_scaled = np.concatenate(preds_scaled_list) # Shape: (num_samples, 1)

    # --- Load Target Scaler and Inverse Transform ---
    target_scaler = joblib.load(target_scaler_local_path)
    preds = target_scaler.inverse_transform(preds_scaled).flatten() # Shape: (num_samples,)

    return preds

def save_predictions(predict_config, ids, preds):
    """
    Saves the predictions to a CSV file locally.

    Args:
        predict_config: Config object containing output paths (results_dir, output_filename).
        ids: Array of IDs corresponding to the predictions.
        preds: Array of predicted values.
    """
    results_dir = predict_config.results_dir
    output_filename = predict_config.output_filename
    os.makedirs(results_dir, exist_ok=True)
    output_path = os.path.join(results_dir, output_filename)

    df_out = pd.DataFrame({
        'ID': ids,
        'puissance_cvac_future': preds
    })
    df_out.to_csv(output_path, index=False)

    return output_path

## 8. Optimization (Optuna)

In [9]:
# --- Functions from optimize.py (adapted for notebook) ---

def objective(trial, base_config):
    """Optuna objective function to train and evaluate the LSTM model."""
    # --- Device ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # --- Suggest Hyperparameters ---
    # Create a trial-specific config based on the base_config
    trial_args = argparse.Namespace(**vars(base_config)) # Copy base config

    # Override with suggested values for LSTM
    trial_args.lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    trial_args.num_layers = trial.suggest_int("num_layers", 1, 4)
    trial_args.batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    trial_args.hidden_size = trial.suggest_int("hidden_size", 32, 128, step=16)
    trial_args.dropout = trial.suggest_float("dropout", 0.0, 0.5) # Make dropout tunable

    test_mae, _ = run_training_trial(trial_args, device, trial_num=trial.number, random_seed=None)

    return test_mae

def run_optimization(config_obj):
    """Sets up and runs the Optuna study for LSTM."""
    # Create or load the study
    # Sampler: TPE is the default and usually works well.
    # Pruner: MedianPruner stops unpromising trials early.
    sampler = optuna.samplers.TPESampler()
    # Configure pruner - n_warmup_steps should relate to how many epochs it takes to get a reasonable loss
    # Adjust warmup steps based on typical convergence time or num_epochs used in optimization
    warmup_epochs = min(10, config_obj.num_epochs // 3) # Example heuristic
    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=warmup_epochs, interval_steps=1)

    study = optuna.create_study(
        study_name=config_obj.study_name,
        storage=config_obj.storage_url,
        direction="minimize", # We want to minimize MAE
        sampler=sampler,
        pruner=pruner,
        load_if_exists=True # Set to True if using storage and want to resume
    )

    # Start the optimization
    study.optimize(lambda trial: objective(trial, config_obj),
                     n_trials=config_obj.n_trials,
                     timeout=None # Adjust timeout (seconds) if needed
                    )

    # --- Results ---
    best_trial = study.best_trial
    print(best_trial.number)
    print(best_trial.value)
    print(best_trial.params)

    return study

## 9. Ensemble Training

In [10]:
def run_ensemble_training(config_obj):
    """
    Simplified version: Runs training trials for multiple seeds defined in config_obj.
    Saves a list of successfully generated model paths locally.

    Args:
        config_obj: Configuration object containing parameters like random_seeds,
                    output_dir, model_paths_filename, and other necessary
                    parameters for run_training_trial.
    """
    base_args = config_obj
    random_seeds = config_obj.random_seeds
    output_dir = base_args.output_dir # Directory to save the paths list
    model_paths_filename = config_obj.model_paths_filename # Filename for the paths list

    # --- Determine Device ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    successful_model_paths = []
    failed_seeds = []

    # --- Loop Through Seeds ---
    for seed in random_seeds:
        # Create a copy of args for this specific run to avoid modification issues
        # run_training_trial expects an args object (like Namespace)
        args_copy = argparse.Namespace(**vars(base_args))

        # Run the simplified training trial
        # It returns (metric, model_path) or (inf, None) on failure
        _, model_path = run_training_trial(args_copy, device, trial_num=None, random_seed=seed)

        if model_path and os.path.exists(model_path):
            # Store relative path if possible, otherwise absolute
            # Using absolute path for simplicity here
            successful_model_paths.append(os.path.abspath(model_path))
        else:
            failed_seeds.append(seed)

    # --- Save Ensemble Info Locally ---
    os.makedirs(output_dir, exist_ok=True)
    paths_filepath = os.path.join(output_dir, model_paths_filename)

    ensemble_info = {
        "successful_model_paths": successful_model_paths,
        "failed_seeds": failed_seeds
    }

    with open(paths_filepath, 'w') as f:
        json.dump(ensemble_info, f, indent=4)

    # Return the collected information
    return ensemble_info


## 10. Pipeline Execution\n
\n
This section demonstrates how to run the different parts of the pipeline (training, prediction, optimization, ensemble) by calling the functions defined above.\n
\n
**Note:** The original `run_pipeline.py` used `subprocess` to execute scripts and captured stdout to find the MLflow Run ID. Here, we will call functions directly. The `run_training_trial` function now prints the Run ID, and we can capture it or retrieve it from MLflow if needed for subsequent steps like prediction.

In [11]:
def run_single_training(config_obj, seed=None):
    """Runs a single LSTM training trial."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    run_args = argparse.Namespace(**vars(config_obj))

    # Assumes run_training_trial now returns paths to model and scalers
    final_metric, model_path, feature_scaler_path, target_scaler_path = run_training_trial(
        run_args, device, trial_num=None, random_seed=seed
    )

    # Return paths needed for prediction
    if model_path:
        return model_path, feature_scaler_path, target_scaler_path, run_args # Return args used
    else:
        return None, None, None, None


def run_prediction_pipeline(config_obj, model_path, feature_scaler_path, target_scaler_path, train_params):
    """Runs the prediction pipeline using saved model and scalers."""
    if not all([model_path, feature_scaler_path, target_scaler_path, train_params]):
        # Cannot proceed without necessary inputs
        return None

    predict_config = argparse.Namespace(**vars(config_obj))

    # Set a simple output filename (can be customized)
    if not predict_config.output_filename or predict_config.output_filename == predict_config.submission_base_filename:
        # Generate a filename based on the model path or a default
        model_basename = os.path.basename(model_path)
        predict_config.output_filename = f"submission_{os.path.splitext(model_basename)[0]}.csv"

    # --- Prepare Test Data ---
    X_test_seq, ids = prepare_test_data(predict_config, train_params, feature_scaler_path)

    # --- Load Model ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = build_model_from_params(train_params)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)

    # --- Run Predictions ---
    preds = run_prediction(model, X_test_seq, target_scaler_path, device)

    # --- Save Submission ---
    submission_path = save_predictions(predict_config, ids, preds)

    return submission_path


In [ ]:
# 1. Define Configuration for Optimization
opt_config = Config() # Start with defaults for LSTM
opt_config.n_trials = 30
opt_config.num_epochs = 20
opt_config.study_name = "hvac_optimization_lstm_workflow_simplified"

# 2. Run Optimization
study = run_optimization(opt_config)

# 3. Get Best Trial Results Directly from Study
best_trial = study.best_trial
model_path = None
feature_scaler_path = None
target_scaler_path = None
train_params = None

if best_trial:
    # Retrieve paths and params saved as user attributes during optimization
    model_path = best_trial.user_attrs.get("model_path")
    feature_scaler_path = best_trial.user_attrs.get("feature_scaler_path")
    target_scaler_path = best_trial.user_attrs.get("target_scaler_path")
    # Retrieve training parameters used for the best trial
    train_params = best_trial.user_attrs.get("train_params") # Should be a dict or Namespace


# 4. Run Prediction using the Best Model Artifacts
if all([model_path, feature_scaler_path, target_scaler_path, train_params]):
    pred_config = Config() # Use a base config for prediction settings
    # Create a unique output filename based on study/trial info if desired
    pred_config.output_filename = f"submission_lstm_optuna_best_trial_{best_trial.number}.csv"

    # Call the modified prediction pipeline
    submission_file = run_prediction_pipeline(
        pred_config,
        model_path,
        feature_scaler_path,
        target_scaler_path,
        train_params # Pass the training parameters
    )


### Example Workflow: Run Ensemble Training

In [ ]:
# 1. Define Configuration for Ensemble
ens_config = Config() # Start with defaults for LSTM
ens_config.random_seeds = [11, 22, 33] # Use a small set of seeds for testing
ens_config.num_epochs = 15 # Maybe fewer epochs for ensemble members
ens_config.ensemble_run_name = "hvac_ensemble_lstm_notebook_test"
ens_config.model_paths_filename = "ensemble_lstm_paths_notebook_test.json"

# 2. Run Ensemble Training
# This will create a parent MLflow run and nested runs for each seed
print("--- Starting Ensemble Training (LSTM Only) ---")
parent_run_id, ensemble_info = run_ensemble_training(ens_config)
print("--- Ensemble Training Finished ---")

print(f"\nEnsemble training finished. Parent Run ID: {parent_run_id}")
print(f"Ensemble Info: {ensemble_info}")

In [ ]:
# --- Optimization Workflow ---
# 1. Define Configuration for Optimization
opt_config = Config() # Start with defaults for LSTM
opt_config.n_trials = 30 # Or a desired number for the pipeline
opt_config.num_epochs = 20 # Or a desired number for the pipeline
opt_config.study_name = "hvac_pipeline_optimization_lstm" # Unique name

# 2. Run Optimization
study = run_optimization(opt_config)

# 3. Get Best Trial Results Directly from Study
best_trial = study.best_trial
optimized_model_info = None

if best_trial:
    print(f"Optimization found best trial: {best_trial.number} with value: {best_trial.value}")
    # Retrieve paths and params saved as user attributes
    optimized_model_info = {
        "model_path": best_trial.user_attrs.get("model_path"),
        "feature_scaler_path": best_trial.user_attrs.get("feature_scaler_path"),
        "target_scaler_path": best_trial.user_attrs.get("target_scaler_path"),
        "train_params": best_trial.user_attrs.get("train_params"),
        "seed": best_trial.user_attrs.get("seed", None) # Store the seed used if available
    }
    # Basic validation
    if not all(optimized_model_info.values()):
        print("Warning: Could not retrieve all artifacts from the best optimization trial.")
        optimized_model_info = None
else:
    print("Optimization did not complete successfully or found no best trial.")

# --- Ensemble Workflow (Using Optimized Model) ---
if optimized_model_info:
    # 4. Define Configuration for Ensemble Training
    ens_config = Config() # Start with defaults
    # Use seeds different from the optimized one, if known, or a fixed set
    # Example: Train 2 additional models alongside the optimized one
    ens_config.random_seeds = [s for s in [11, 22, 33] if s != optimized_model_info.get("seed")]
    if not ens_config.random_seeds: # Ensure there are seeds if the optimized one was in the list
         ens_config.random_seeds = [44, 55] # Add fallback seeds
    print(f"Optimized model seed: {optimized_model_info.get('seed')}. Using seeds {ens_config.random_seeds} for additional ensemble members.")

    ens_config.num_epochs = optimized_model_info["train_params"].get("num_epochs", 15) # Use optimized epochs or default
    ens_config.ensemble_run_name = "hvac_pipeline_ensemble_with_opt"
    ens_config.model_paths_filename = "ensemble_pipeline_paths.json" # Unique name

    # 5. Run Ensemble Training, providing the optimized model info
    print("--- Starting Ensemble Training Phase (incorporating optimized model) ---")
    # NOTE: run_ensemble_training needs modification to accept optimized_model_info
    parent_run_id, ensemble_info = run_ensemble_training(
        ens_config,
        optimized_model_info=optimized_model_info # Pass the optimized model details
    )
    print("--- Ensemble Training Finished ---")

    print(f"Ensemble training finished. Parent Run ID: {parent_run_id}")
    print(f"Ensemble Info: {ensemble_info}") # This should now contain paths for all models

    # --- Prediction Workflow (Using Ensemble) ---
    # 6. Define Configuration for Prediction
    pred_config = Config()
    pred_config.output_filename = f"submission_pipeline_ensemble_{parent_run_id}.csv"

    # 7. Run Prediction using the Ensemble Artifacts
    print("--- Starting Prediction Phase (using ensemble) ---")
    # NOTE: This likely requires a new function run_ensemble_prediction
    #       or modification to run_prediction_pipeline to handle ensemble_info
    submission_file = run_ensemble_prediction( # Assuming this function exists/will be created
        pred_config,
        ensemble_info # Pass the collected info for all ensemble models
    )
    print("--- Prediction Finished ---")
    print(f"Submission file created: {submission_file}")

else:
    print("Skipping Ensemble and Prediction phases due to optimization issues.")



In [17]:
def run_ensemble_prediction(predict_config, ensemble_info):
    """
    Runs predictions using an ensemble of models and averages the results.

    Args:
        predict_config: Config object containing prediction settings (e.g., test data path, output filename).
        ensemble_info: Dictionary where keys are model identifiers and values are dicts
                       containing 'model_path', 'feature_scaler_path', 'target_scaler_path',
                       and 'train_params' (as Namespace or dict) for each ensemble member.

    Returns:
        Path to the final submission file, or None if prediction fails.
    """
    print(f"--- Running Ensemble Prediction ---")
    if not ensemble_info:
        print("Error: ensemble_info dictionary is empty. Cannot perform ensemble prediction.")
        return None

    all_predictions = []
    ids = None # To store IDs from the first model's data prep

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Create a Namespace from predict_config if it's not already one
    pred_args = argparse.Namespace(**vars(predict_config))

    for model_id, model_data in ensemble_info.items():
        print(f"\n--- Processing ensemble member: {model_id} ---")

        model_path = model_data.get("model_path")
        feature_scaler_path = model_data.get("feature_scaler_path")
        target_scaler_path = model_data.get("target_scaler_path")
        train_params = model_data.get("train_params") # Expected to be Namespace or dict

        if not all([model_path, feature_scaler_path, target_scaler_path, train_params]):
            print(f"Warning: Missing required artifact path or train_params for model {model_id}. Skipping this model.")
            continue

        # Ensure train_params is a Namespace for consistent access
        if isinstance(train_params, dict):
             current_train_params = argparse.Namespace(**train_params)
        else:
             current_train_params = train_params # Assume it's already a Namespace

        try:
            # 1. Prepare Test Data (using this model's scaler and train_params)
            # Note: Assumes prepare_test_data can handle local paths directly
            print(f"Preparing test data using scaler: {os.path.basename(feature_scaler_path)}")
            # Pass pred_args (from predict_config), current_train_params, and the specific feature_scaler_path
            X_test_seq, current_ids = prepare_test_data(pred_args, current_train_params, feature_scaler_path)

            if ids is None:
                ids = current_ids # Store IDs from the first successful model
            elif not np.array_equal(ids, current_ids):
                print("Warning: ID mismatch between ensemble members. Using IDs from the first model.")
                # Potentially raise an error or handle differently if IDs *should* be identical

            # 2. Load Model
            print(f"Loading model: {os.path.basename(model_path)}")
            # Assumes build_model_from_params uses train_params to create the correct architecture
            model = build_model_from_params(current_train_params) # Needs train_params to know architecture
            model.load_state_dict(torch.load(model_path, map_location=device))
            model.to(device)
            model.eval()

            # 3. Run Prediction (for this model)
            print(f"Running prediction for model {model_id}...")
            preds_scaled_list = []
            # Reusing batch prediction logic from run_prediction
            X_tensor = torch.from_numpy(X_test_seq).float()
            pred_dataset = TensorDataset(X_tensor)
            pred_loader = DataLoader(pred_dataset, batch_size=256, shuffle=False) # Use a suitable batch size

            with torch.no_grad():
                 for Xb, in tqdm(pred_loader, desc=f"Predicting {model_id}", leave=False):
                    Xb = Xb.to(device)
                    batch_preds_scaled = model(Xb).cpu().numpy()
                    preds_scaled_list.append(batch_preds_scaled)

            preds_scaled = np.concatenate(preds_scaled_list)

            # 4. Inverse Transform Predictions
            print(f"Loading target scaler: {os.path.basename(target_scaler_path)}")
            target_scaler = joblib.load(target_scaler_path)
            preds = target_scaler.inverse_transform(preds_scaled).flatten()
            all_predictions.append(preds)
            print(f"Successfully generated predictions for model {model_id}.")

        except Exception as e:
            print(f"Error processing model {model_id}: {e}")
            import traceback
            traceback.print_exc()
            # Decide if you want to continue with other models or fail the ensemble
            # continue # Option: Skip failed model

    # 5. Average Predictions
    if not all_predictions:
        print("Error: No predictions were generated from any ensemble member.")
        return None
    if ids is None:
        print("Error: Could not retrieve IDs from any ensemble member's data preparation.")
        return None

    print(f"\nAveraging predictions from {len(all_predictions)} ensemble members.")
    # Stack predictions into a 2D array (models x samples) and average across models (axis=0)
    final_preds = np.mean(np.vstack(all_predictions), axis=0)

    # 6. Save Final Submission
    print("Saving final averaged predictions...")
    # Assumes save_predictions function exists and uses pred_args for output dir/filename
    submission_path = save_predictions(pred_args, ids, final_preds)

    print(f"--- Ensemble Prediction Finished. Final submission: {submission_path} ---")
    return submission_path

In [ ]:
a = run_ensemble_prediction(Config(), )